# Data Pipeline — Amazon Reviews 2023 (Appliances)
Exploratory analysis on a streamed sample, followed by schema design and ingestion.

> **Note:** `datasets` >= 3.x removed support for dataset loading scripts
> (`RuntimeError: Dataset scripts are no longer supported`). The raw `.jsonl` file is
> streamed directly via the `json` builder with an `hf://` path instead.

In [4]:
from datasets import load_dataset

# datasets >= 3.x no longer runs dataset scripts; stream the raw jsonl file directly
reviews_stream = load_dataset(
    "json",
    data_files="hf://datasets/McAuley-Lab/Amazon-Reviews-2023/raw/review_categories/Appliances.jsonl",
    split="train",
    streaming=True,
)

sample = []
for i, record in enumerate(reviews_stream):
    if i >= 5000:
        break
    sample.append(record)

print(f"Collected {len(sample)} records")
print(sample[0])

Collected 5000 records
{'rating': 5.0, 'title': 'Work great', 'text': 'work great. use a new one every month', 'images': [], 'asin': 'B01N0TQ0OH', 'parent_asin': 'B01N0TQ0OH', 'user_id': 'AGKHLEW2SOWHNMFQIJGBECAF7INQ', 'timestamp': 1519317108692, 'helpful_vote': 0, 'verified_purchase': True}


In [5]:
import pandas as pd
df = pd.DataFrame(sample)

print("Shape:", df.shape)
print("\n--- Null counts ---")
print(df.isnull().sum())
print("\n--- Rating distribution ---")
print(df["rating"].value_counts().sort_index())
print("\n--- Timestamp range ---")
ts = pd.to_datetime(df["timestamp"], unit="ms")
print("min:", ts.min(), "| max:", ts.max())
print("\n--- Text length (chars) ---")
print(df["text"].str.len().describe().round(1))

Shape: (5000, 10)

--- Null counts ---
rating               0
title                0
text                 0
images               0
asin                 0
parent_asin          0
user_id              0
timestamp            0
helpful_vote         0
verified_purchase    0
dtype: int64

--- Rating distribution ---
rating
1.0     376
2.0     170
3.0     293
4.0     582
5.0    3579
Name: count, dtype: int64

--- Timestamp range ---
min: 2007-03-29 23:05:20 | max: 2023-03-17 07:53:04.630000

--- Text length (chars) ---
count     5000.0
mean       248.8
std        454.4
min          1.0
25%         47.0
50%        116.0
75%        279.0
max      11328.0
Name: text, dtype: float64


## Sample EDA — findings

- **Nulls:** none in the 5k sample; ingest still handles nulls defensively (sample ≠ full set).
- **Rating distribution:** J-shaped (72% five-star, 7.5% one-star) — typical for Amazon reviews.
  Averages alone are misleading; distribution will be reported alongside.
- **Timestamp:** 13-digit epoch **milliseconds** (verified: 2007–2023 range after `unit="ms"`).
  Will be converted to seconds (`// 1000`) at ingestion.
- **Text length:** heavily right-skewed (median 116 chars, max 11,328). Most reviews fit a
  single chunk; a long tail needs chunking. Reviews under ~20 chars will be excluded from
  the retrieval corpus.